# Böbrek Hastalığı Sınıflandırması - MLP ve Aktivasyon Fonksiyonları Analizi

Bu notebook'ta böbrek hastalığı veri setini kullanarak:
1. Veri setini kavrama ve EDA
2. Görselleştirme
3. Feature Importance
4. Farklı aktivasyon fonksiyonlarını deneme
5. Farklı epoch sayılarını deneme
6. Farklı optimizerları deneme

işlemlerini gerçekleştireceğiz.

## 1. Kütüphanelerin İçe Aktarılması

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML kütüphaneleri
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, SelectKBest, f_classif

# Neural Network
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU, PReLU, ELU
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Adagrad, Nadam, Adamax
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler

# Ayarlar
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Kütüphaneler başarıyla yüklendi!")
print(f"TensorFlow Version: {tf.__version__}")

ModuleNotFoundError: No module named 'tensorflow'

## 2. Veri Setinin Yüklenmesi ve İlk İnceleme

In [ ]:
# Veri setini yükle
df = pd.read_csv('kidney_disease_turkce.csv')

print("="*60)
print("VERİ SETİ BİLGİLERİ")
print("="*60)
print(f"Veri Seti Boyutu: {df.shape}")
print(f"Satır Sayısı: {df.shape[0]}")
print(f"Sütun Sayısı: {df.shape[1]}")
print("\n" + "="*60)
print("İLK 10 SATIR")
print("="*60)
df.head(10)

In [ ]:
print("="*60)
print("VERİ SETİ ÖZETİ")
print("="*60)
df.info()

In [ ]:
print("="*60)
print("İSTATİSTİKSEL ÖZET")
print("="*60)
df.describe().T

In [ ]:
print("="*60)
print("EKSİK DEĞER ANALİZİ")
print("="*60)
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Eksik Değer Sayısı': missing,
    'Yüzde (%)': missing_percent
})
missing_df = missing_df[missing_df['Eksik Değer Sayısı'] > 0].sort_values('Eksik Değer Sayısı', ascending=False)
print(missing_df)
print(f"\nToplam Eksik Değer: {df.isnull().sum().sum()}")

In [ ]:
print("="*60)
print("HEDEF DEĞİŞKEN DAĞILIMI (siniflandirma)")
print("="*60)
print(df['siniflandirma'].value_counts())
print(f"\nYüzdesel Dağılım:\n{df['siniflandirma'].value_counts(normalize=True) * 100}")

In [ ]:
# Kategorik ve sayısal sütunları ayır
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("="*60)
print("SÜTUN TİPLERİ")
print("="*60)
print(f"Kategorik Sütunlar ({len(categorical_cols)}): {categorical_cols}")
print(f"\nSayısal Sütunlar ({len(numerical_cols)}): {numerical_cols}")

# ID sütununu çıkar
if 'id' in numerical_cols:
    numerical_cols.remove('id')
if 'siniflandirma' in categorical_cols:
    categorical_cols.remove('siniflandirma')

## 3. Keşifsel Veri Analizi (EDA) ve Görselleştirme

In [ ]:
# Hedef değişken dağılımı
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

df['siniflandirma'].value_counts().plot.pie(
    autopct='%1.1f%%', 
    ax=axes[0], 
    colors=['#ff6b6b', '#4ecdc4'],
    explode=(0.05, 0),
    shadow=True
)
axes[0].set_title('Hedef Değişken Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_ylabel('')

sns.countplot(data=df, x='siniflandirma', ax=axes[1], palette=['#ff6b6b', '#4ecdc4'])
axes[1].set_title('Hasta Sayısı', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Sınıflandırma')
axes[1].set_ylabel('Sayı')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}', 
                     (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha='center', va='center', 
                     xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
# Sayısal değişkenlerin dağılımı
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i, col in enumerate(numerical_cols[:16]):
    if col in df.columns:
        sns.histplot(data=df, x=col, kde=True, ax=axes[i], color='steelblue', alpha=0.7)
        axes[i].set_title(f'{col.replace("_", " ").title()} Dağılımı', fontsize=10)
        axes[i].set_xlabel('')
        axes[i].set_ylabel('')

# Boş subplotları gizle
for i in range(len(numerical_cols[:16]), len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Sayısal Değişkenlerin Dağılımı', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot ile aykırı değer analizi
fig, axes = plt.subplots(4, 4, figsize=(18, 16))
axes = axes.flatten()

for i, col in enumerate(numerical_cols[:16]):
    if col in df.columns:
        sns.boxplot(data=df, y=col, ax=axes[i], color='lightcoral')
        axes[i].set_title(f'{col.replace("_", " ").title()}', fontsize=10)
        axes[i].set_ylabel('')

# Boş subplotları gizle
for i in range(len(numerical_cols[:16]), len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Aykırı Değer Analizi (Boxplot)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Hedef değişkene göre sayısal değişkenlerin dağılımı
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

important_nums = ['yas', 'kan_basinci', 'ozgul_agirlik', 'kan_ure', 'serum_kreatinin', 
                 'hemoglobin', 'paketli_hucre_hacmi', 'beyaz_kan_hucresi_sayisi', 'kirmizi_kan_hucresi_sayisi']

for i, col in enumerate(important_nums):
    if col in df.columns:
        for sinif in df['siniflandirma'].unique():
            sns.kdeplot(data=df[df['siniflandirma'] == sinif], x=col, 
                       label=sinif, ax=axes[i], fill=True, alpha=0.3)
        axes[i].set_title(f'{col.replace("_", " ").title()}', fontsize=10)
        axes[i].legend()

plt.suptitle('Sınıflara Göre Değişken Dağılımları', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Kategorik değişkenlerin analizi
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

important_cats = ['kirmizi_kan_hucresi', 'irin_hucresi', 'irin_hucresi_kumesi', 'bakteri', 
                 'hipertansiyon', 'diyabet', 'koroner_arter_hastaligi', 'istah', 'ayak_odemi']

for i, col in enumerate(important_cats):
    if col in df.columns:
        ct = pd.crosstab(df[col], df['siniflandirma'], normalize='index') * 100
        ct.plot(kind='bar', stacked=True, ax=axes[i], color=['#ff6b6b', '#4ecdc4'])
        axes[i].set_title(f'{col.replace("_", " ").title()}', fontsize=10)
        axes[i].set_ylabel('Yüzde (%)')
        axes[i].legend(title='Sınıf')
        axes[i].tick_params(axis='x', rotation=45)

# Boş subplotları gizle
for i in range(len(important_cats), len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Kategorik Değişkenlerin Sınıflara Göre Dağılımı', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Korelasyon Analizi

In [ ]:
# Sayısal değişkenler için korelasyon matrisi
numerical_df = df[numerical_cols].copy()
numerical_df['siniflandirma_encoded'] = df['siniflandirma'].map({'ckd': 1, 'notckd': 0})

plt.figure(figsize=(14, 12))
correlation_matrix = numerical_df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, mask=mask, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Korelasyon Matrisi', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Hedef değişken ile korelasyonlar
target_corr = numerical_df.corr()['siniflandirma_encoded'].sort_values(ascending=False)
plt.figure(figsize=(10, 8))
target_corr.drop('siniflandirma_encoded').plot(kind='barh', color='steelblue')
plt.title('Hedef Değişken ile Korelasyonlar', fontsize=14, fontweight='bold')
plt.xlabel('Korelasyon Katsayısı')
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5. Veri Ön İşleme

In [ ]:
# Veri setini kopyala
df_clean = df.copy()

# ID sütununu kaldır
if 'id' in df_clean.columns:
    df_clean = df_clean.drop('id', axis=1)

# Hedef değişkeni encode et
label_encoder = LabelEncoder()
df_clean['siniflandirma'] = label_encoder.fit_transform(df_clean['siniflandirma'])
print("Sınıf Etiketleri:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

# Kategorik değişkenleri encode et
categorical_features = df_clean.select_dtypes(include=['object']).columns.tolist()
print(f"\nKategorik değişkenler: {categorical_features}")

for col in categorical_features:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0] if not df_clean[col].mode().empty else 'unknown')
    df_clean[col] = label_encoder.fit_transform(df_clean[col].astype(str))

# Sayısal değişkenlerdeki eksik değerleri doldur
numerical_features = df_clean.select_dtypes(include=[np.number]).columns.tolist()
numerical_features.remove('siniflandirma')

imputer = SimpleImputer(strategy='median')
df_clean[numerical_features] = imputer.fit_transform(df_clean[numerical_features])

print(f"\nEksik değer sayısı: {df_clean.isnull().sum().sum()}")
print(f"Temizlenmiş veri seti boyutu: {df_clean.shape}")

In [ ]:
# Özellikler ve hedef değişken
X = df_clean.drop('siniflandirma', axis=1)
y = df_clean['siniflandirma']

print(f"Özellikler boyutu: {X.shape}")
print(f"Hedef değişken boyutu: {y.shape}")
print(f"Sınıf dağılımı:\n{y.value_counts()}")

In [ ]:
# Veriyi ölçeklendir
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Eğitim seti boyutu: {X_train.shape}")
print(f"Test seti boyutu: {X_test.shape}")
print(f"\nEğitim seti sınıf dağılımı:\n{y_train.value_counts()}")
print(f"\nTest seti sınıf dağılımı:\n{y_test.value_counts()}")

## 6. Feature Importance

In [ ]:
# Random Forest ile Feature Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Feature importance
feature_importance = pd.DataFrame({
    'Özellik': X.columns,
    'Önem Skoru': rf.feature_importances_
}).sort_values('Önem Skoru', ascending=False)

print("="*60)
print("FEATURE IMPORTANCE (Random Forest)")
print("="*60)
print(feature_importance.to_string(index=False))

In [ ]:
# Feature Importance Görselleştirme
plt.figure(figsize=(12, 10))
sns.barplot(data=feature_importance.head(15), x='Önem Skoru', y='Özellik', palette='viridis')
plt.title('En Önemli 15 Özellik (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Önem Skoru')
plt.tight_layout()
plt.show()

In [ ]:
# Mutual Information
mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
mi_importance = pd.DataFrame({
    'Özellik': X.columns,
    'MI Skoru': mi_scores
}).sort_values('MI Skoru', ascending=False)

print("="*60)
print("MUTUAL INFORMATION SCORES")
print("="*60)
print(mi_importance.to_string(index=False))

In [ ]:
# Mutual Information Görselleştirme
plt.figure(figsize=(12, 10))
sns.barplot(data=mi_importance.head(15), x='MI Skoru', y='Özellik', palette='plasma')
plt.title('En Önemli 15 Özellik (Mutual Information)', fontsize=14, fontweight='bold')
plt.xlabel('Mutual Information Skoru')
plt.tight_layout()
plt.show()

In [ ]:
# ANOVA F-test
f_scores, p_values = f_classif(X_train, y_train)
anova_importance = pd.DataFrame({
    'Özellik': X.columns,
    'F-Skoru': f_scores,
    'p-değeri': p_values
}).sort_values('F-Skoru', ascending=False)

print("="*60)
print("ANOVA F-TEST SONUÇLARI")
print("="*60)
print(anova_importance.to_string(index=False))

In [ ]:
# ANOVA Görselleştirme
plt.figure(figsize=(12, 10))
sns.barplot(data=anova_importance.head(15), x='F-Skoru', y='Özellik', palette='magma')
plt.title('En Önemli 15 Özellik (ANOVA F-Test)', fontsize=14, fontweight='bold')
plt.xlabel('F-Skoru')
plt.tight_layout()
plt.show()

In [ ]:
# En önemli özellikleri seç
top_features = feature_importance.head(15)['Özellik'].values
print(f"Seçilen {len(top_features)} özellik:")
for i, feat in enumerate(top_features, 1):
    print(f"{i}. {feat}")

# Sadece en önemli özelliklerle yeni veri seti
X_train_selected = X_train[top_features]
X_test_selected = X_test[top_features]

print(f"\nYeni eğitim seti boyutu: {X_train_selected.shape}")
print(f"Yeni test seti boyutu: {X_test_selected.shape}")

## 7. MLP Model Denemeleri - Aktivasyon Fonksiyonları

Bu bölümde farklı aktivasyon fonksiyonlarını deneyerek sonuçları karşılaştıracağız.

In [ ]:
# Sonuçları saklamak için sözlük
activation_results = {}
activation_histories = {}

# Aktivasyon fonksiyonları
activations = ['relu', 'sigmoid', 'tanh']

def create_model(activation='relu', learning_rate=0.001):
    model = Sequential([
        Dense(64, input_dim=X_train_selected.shape[1], activation=activation),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation=activation),
        BatchNormalization(),
        Dropout(0.2),
        Dense(16, activation=activation),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

print("MLP modelleri eğitiliyor...")
for activation in activations:
    print(f"\n{'='*50}")
    print(f"Aktivasyon Fonksiyonu: {activation.upper()}")
    print(f"{'='*50}")
    
    model = create_model(activation=activation)
    
    early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)
    
    history = model.fit(
        X_train_selected, y_train,
        epochs=100,
        batch_size=16,
        validation_data=(X_test_selected, y_test),
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )
    
    # Değerlendirme
    train_loss, train_acc, train_auc = model.evaluate(X_train_selected, y_train, verbose=0)
    test_loss, test_acc, test_auc = model.evaluate(X_test_selected, y_test, verbose=0)
    
    # Tahminler
    y_pred_prob = model.predict(X_test_selected, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    # Sonuçları kaydet
    activation_results[activation] = {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_auc': train_auc,
        'test_auc': test_auc,
        'epochs_run': len(history.history['loss'])
    }
    activation_histories[activation] = history
    
    print(f"Eğitim Doğruluk: {train_acc:.4f}")
    print(f"Test Doğruluk: {test_acc:.4f}")
    print(f"Eğitim AUC: {train_auc:.4f}")
    print(f"Test AUC: {test_auc:.4f}")
    print(f"Çalışan Epoch: {len(history.history['loss'])}")

In [ ]:
# Aktivasyon fonksiyonu karşılaştırma tablosu
comparison_df = pd.DataFrame(activation_results).T
print("="*70)
print("AKTİVASYON FONKSİYONU KARŞILAŞTIRMASI")
print("="*70)
print(comparison_df.round(4))

In [ ]:
# Aktivasyon fonksiyonu performans görselleştirme
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

metrics = ['train_accuracy', 'test_accuracy', 'train_auc', 'test_auc']
titles = ['Eğitim Doğruluk', 'Test Doğruluk', 'Eğitim AUC', 'Test AUC']

for i, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[i // 3, i % 3]
    values = [activation_results[act][metric] for act in activations]
    bars = ax.bar(activations, values, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Skor')
    ax.set_ylim(0, 1)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)

# Epoch karşılaştırması
ax = axes[1, 2]
epochs = [activation_results[act]['epochs_run'] for act in activations]
bars = ax.bar(activations, epochs, color=['#9b59b6', '#f39c12', '#1abc9c'])
ax.set_title('Çalışan Epoch Sayısı', fontsize=12, fontweight='bold')
ax.set_ylabel('Epoch')
for bar, val in zip(bars, epochs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f'{val}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Aktivasyon Fonksiyonu Performans Karşılaştırması', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Eğitim geçmişi görselleştirme
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for i, (activation, history) in enumerate(activation_histories.items()):
    # Loss
    axes[0, i].plot(history.history['loss'], label='Eğitim', linewidth=2)
    axes[0, i].plot(history.history['val_loss'], label='Doğrulama', linewidth=2)
    axes[0, i].set_title(f'{activation.upper()} - Loss', fontsize=12, fontweight='bold')
    axes[0, i].set_xlabel('Epoch')
    axes[0, i].set_ylabel('Loss')
    axes[0, i].legend()
    axes[0, i].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1, i].plot(history.history['accuracy'], label='Eğitim', linewidth=2)
    axes[1, i].plot(history.history['val_accuracy'], label='Doğrulama', linewidth=2)
    axes[1, i].set_title(f'{activation.upper()} - Accuracy', fontsize=12, fontweight='bold')
    axes[1, i].set_xlabel('Epoch')
    axes[1, i].set_ylabel('Accuracy')
    axes[1, i].legend()
    axes[1, i].grid(True, alpha=0.3)

plt.suptitle('Eğitim Geçmişleri', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Epoch Sayısı Denemeleri

In [ ]:
# Farklı epoch sayılarını dene
epoch_values = [20, 50, 100, 150, 200]
epoch_results = {}
epoch_histories = {}

print("Epoch sayısı denemeleri yapılıyor...")
for epochs in epoch_values:
    print(f"\n{'='*50}")
    print(f"Epoch Sayısı: {epochs}")
    print(f"{'='*50}")
    
    model = create_model(activation='relu')
    
    history = model.fit(
        X_train_selected, y_train,
        epochs=epochs,
        batch_size=16,
        validation_data=(X_test_selected, y_test),
        verbose=0
    )
    
    train_loss, train_acc, train_auc = model.evaluate(X_train_selected, y_train, verbose=0)
    test_loss, test_acc, test_auc = model.evaluate(X_test_selected, y_test, verbose=0)
    
    epoch_results[epochs] = {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_auc': train_auc,
        'test_auc': test_auc,
        'final_train_loss': history.history['loss'][-1],
        'final_val_loss': history.history['val_loss'][-1]
    }
    epoch_histories[epochs] = history
    
    print(f"Eğitim Doğruluk: {train_acc:.4f}")
    print(f"Test Doğruluk: {test_acc:.4f}")
    print(f"Eğitim AUC: {train_auc:.4f}")
    print(f"Test AUC: {test_auc:.4f}")
    print(f"Final Train Loss: {history.history['loss'][-1]:.4f}")
    print(f"Final Val Loss: {history.history['val_loss'][-1]:.4f}")

In [ ]:
# Epoch karşılaştırma tablosu
epoch_comparison_df = pd.DataFrame(epoch_results).T
print("="*70)
print("EPOCH SAYISI KARŞILAŞTIRMASI")
print("="*70)
print(epoch_comparison_df.round(4))

In [ ]:
# Epoch performans görselleştirme
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Accuracy
axes[0, 0].plot(epoch_values, [epoch_results[e]['train_accuracy'] for e in epoch_values], 
                'o-', label='Eğitim', linewidth=2, markersize=8)
axes[0, 0].plot(epoch_values, [epoch_results[e]['test_accuracy'] for e in epoch_values], 
                's-', label='Test', linewidth=2, markersize=8)
axes[0, 0].set_title('Epoch vs Accuracy', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch Sayısı')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# AUC
axes[0, 1].plot(epoch_values, [epoch_results[e]['train_auc'] for e in epoch_values], 
                'o-', label='Eğitim', linewidth=2, markersize=8)
axes[0, 1].plot(epoch_values, [epoch_results[e]['test_auc'] for e in epoch_values], 
                's-', label='Test', linewidth=2, markersize=8)
axes[0, 1].set_title('Epoch vs AUC', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch Sayısı')
axes[0, 1].set_ylabel('AUC')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Loss
axes[1, 0].plot(epoch_values, [epoch_results[e]['final_train_loss'] for e in epoch_values], 
                'o-', label='Eğitim', linewidth=2, markersize=8)
axes[1, 0].plot(epoch_values, [epoch_results[e]['final_val_loss'] for e in epoch_values], 
                's-', label='Doğrulama', linewidth=2, markersize=8)
axes[1, 0].set_title('Epoch vs Loss', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch Sayısı')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Overfitting analizi
overfitting = [epoch_results[e]['train_accuracy'] - epoch_results[e]['test_accuracy'] for e in epoch_values]
colors = ['green' if x < 0.05 else 'orange' if x < 0.1 else 'red' for x in overfitting]
axes[1, 1].bar(range(len(epoch_values)), overfitting, color=colors)
axes[1, 1].set_title('Overfitting Analizi (Train - Test Acc)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch Sayısı')
axes[1, 1].set_ylabel('Accuracy Farkı')
axes[1, 1].set_xticks(range(len(epoch_values)))
axes[1, 1].set_xticklabels(epoch_values)
axes[1, 1].axhline(y=0.05, color='red', linestyle='--', alpha=0.5, label='Threshold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Epoch Performans Analizi', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Tüm epoch değerlerinin eğitim geçmişleri
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (epochs, history) in enumerate(epoch_histories.items()):
    axes[i].plot(history.history['loss'], label='Eğitim', linewidth=2)
    axes[i].plot(history.history['val_loss'], label='Doğrulama', linewidth=2)
    axes[i].set_title(f'{epochs} Epoch', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

axes[-1].set_visible(False)
plt.suptitle('Farklı Epoch Sayıları İçin Eğitim Geçmişi', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Optimizer Denemeleri

In [ ]:
# Farklı optimizerları dene
def create_model_with_optimizer(optimizer_name='adam', learning_rate=0.001):
    model = Sequential([
        Dense(64, input_dim=X_train_selected.shape[1], activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    
    # Optimizer seçimi
    if optimizer_name == 'adam':
        opt = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        opt = SGD(learning_rate=learning_rate, momentum=0.9)
    elif optimizer_name == 'rmsprop':
        opt = RMSprop(learning_rate=learning_rate)
    elif optimizer_name == 'adagrad':
        opt = Adagrad(learning_rate=learning_rate)
    elif optimizer_name == 'nadam':
        opt = keras.optimizers.Nadam(learning_rate=learning_rate)
    elif optimizer_name == 'adamax':
        opt = keras.optimizers.Adamax(learning_rate=learning_rate)
    else:
        opt = Adam(learning_rate=learning_rate)
    
    model.compile(
        optimizer=opt,
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

optimizers = ['adam', 'sgd', 'rmsprop', 'adagrad', 'nadam', 'adamax']
optimizer_results = {}
optimizer_histories = {}

print("Optimizer denemeleri yapılıyor...")
for opt_name in optimizers:
    print(f"\n{'='*50}")
    print(f"Optimizer: {opt_name.upper()}")
    print(f"{'='*50}")
    
    model = create_model_with_optimizer(optimizer_name=opt_name)
    
    early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6)
    
    history = model.fit(
        X_train_selected, y_train,
        epochs=100,
        batch_size=16,
        validation_data=(X_test_selected, y_test),
        callbacks=[early_stop, reduce_lr],
        verbose=0
    )
    
    train_loss, train_acc, train_auc = model.evaluate(X_train_selected, y_train, verbose=0)
    test_loss, test_acc, test_auc = model.evaluate(X_test_selected, y_test, verbose=0)
    
    optimizer_results[opt_name] = {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_auc': train_auc,
        'test_auc': test_auc,
        'epochs_run': len(history.history['loss'])
    }
    optimizer_histories[opt_name] = history
    
    print(f"Eğitim Doğruluk: {train_acc:.4f}")
    print(f"Test Doğruluk: {test_acc:.4f}")
    print(f"Eğitim AUC: {train_auc:.4f}")
    print(f"Test AUC: {test_auc:.4f}")
    print(f"Çalışan Epoch: {len(history.history['loss'])}")

In [ ]:
# Optimizer karşılaştırma tablosu
optimizer_comparison_df = pd.DataFrame(optimizer_results).T
print("="*70)
print("OPTİMİZER KARŞILAŞTIRMASI")
print("="*70)
print(optimizer_comparison_df.round(4))

In [ ]:
# Optimizer performans görselleştirme
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Accuracy
x_pos = np.arange(len(optimizers))
width = 0.35
axes[0, 0].bar(x_pos - width/2, [optimizer_results[o]['train_accuracy'] for o in optimizers], 
              width, label='Eğitim', color='#3498db')
axes[0, 0].bar(x_pos + width/2, [optimizer_results[o]['test_accuracy'] for o in optimizers], 
              width, label='Test', color='#e74c3c')
axes[0, 0].set_title('Optimizer - Accuracy', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Optimizer')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(optimizers, rotation=45)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# AUC
axes[0, 1].bar(x_pos - width/2, [optimizer_results[o]['train_auc'] for o in optimizers], 
              width, label='Eğitim', color='#3498db')
axes[0, 1].bar(x_pos + width/2, [optimizer_results[o]['test_auc'] for o in optimizers], 
              width, label='Test', color='#e74c3c')
axes[0, 1].set_title('Optimizer - AUC', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Optimizer')
axes[0, 1].set_ylabel('AUC')
axes[0, 1].set_xticks(x_pos)
axes[0, 1].set_xticklabels(optimizers, rotation=45)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Epoch
axes[1, 0].bar(optimizers, [optimizer_results[o]['epochs_run'] for o in optimizers], color='#9b59b6')
axes[1, 0].set_title('Optimizer - Çalışan Epoch', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Optimizer')
axes[1, 0].set_ylabel('Epoch Sayısı')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3)

# Overfitting
overfitting = [optimizer_results[o]['train_accuracy'] - optimizer_results[o]['test_accuracy'] for o in optimizers]
colors = ['green' if x < 0.05 else 'orange' if x < 0.1 else 'red' for x in overfitting]
axes[1, 1].bar(optimizers, overfitting, color=colors)
axes[1, 1].set_title('Optimizer - Overfitting (Train-Test)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Optimizer')
axes[1, 1].set_ylabel('Accuracy Farkı')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].axhline(y=0.05, color='red', linestyle='--', alpha=0.5)
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Optimizer Performans Analizi', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Optimizer eğitim geçmişleri
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (opt_name, history) in enumerate(optimizer_histories.items()):
    axes[i].plot(history.history['loss'], label='Eğitim', linewidth=2)
    axes[i].plot(history.history['val_loss'], label='Doğrulama', linewidth=2)
    axes[i].set_title(f'{opt_name.upper()}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Optimizer Eğitim Geçmişleri', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. En İyi Model ile Sonuçlar

In [ ]:
# En iyi modeli bul
best_optimizer = max(optimizer_results, key=lambda x: optimizer_results[x]['test_accuracy'])
print(f"En iyi optimizer: {best_optimizer.upper()}")
print(f"Test Accuracy: {optimizer_results[best_optimizer]['test_accuracy']:.4f}")
print(f"Test AUC: {optimizer_results[best_optimizer]['test_auc']:.4f}")

# En iyi modeli oluştur ve eğit
best_model = create_model_with_optimizer(optimizer_name=best_optimizer)
history = best_model.fit(
    X_train_selected, y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_test_selected, y_test),
    verbose=0
)

In [ ]:
# Detaylı değerlendirme
y_pred_prob = best_model.predict(X_test_selected, verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int)

print("="*60)
print("EN İYİ MODEL DETAYLI DEĞERLENDİRME")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not CKD', 'CKD']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not CKD', 'CKD'], 
            yticklabels=['Not CKD', 'CKD'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - En İyi Model', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

## 11. Özet Karşılaştırma

In [ ]:
# Tüm denemelerin özeti
summary_data = []

# Aktivasyon fonksiyonları
for act, res in activation_results.items():
    summary_data.append({
        'Kategori': 'Aktivasyon',
        'Parametre': act.upper(),
        'Test Accuracy': res['test_accuracy'],
        'Test AUC': res['test_auc']
    })

# Epoch sayıları
for epoch, res in epoch_results.items():
    summary_data.append({
        'Kategori': 'Epoch',
        'Parametre': f'{epoch} ep',
        'Test Accuracy': res['test_accuracy'],
        'Test AUC': res['test_auc']
    })

# Optimizerlar
for opt, res in optimizer_results.items():
    summary_data.append({
        'Kategori': 'Optimizer',
        'Parametre': opt.upper(),
        'Test Accuracy': res['test_accuracy'],
        'Test AUC': res['test_auc']
    })

summary_df = pd.DataFrame(summary_data)

# En iyi sonuçları göster
print("="*70)
print("TÜM DENEMELERİN ÖZETİ")
print("="*70)
print(summary_df.sort_values('Test Accuracy', ascending=False).head(15).to_string(index=False))

In [ ]:
# Kategoriye göre en iyi sonuçlar
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Aktivasyon
act_data = summary_df[summary_df['Kategori'] == 'Aktivasyon'].sort_values('Test Accuracy')
axes[0].barh(act_data['Parametre'], act_data['Test Accuracy'], color='steelblue')
axes[0].set_title('Aktivasyon Fonksiyonları', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Test Accuracy')

# Epoch
epoch_data = summary_df[summary_df['Kategori'] == 'Epoch'].sort_values('Test Accuracy')
axes[1].barh(epoch_data['Parametre'], epoch_data['Test Accuracy'], color='coral')
axes[1].set_title('Epoch Sayıları', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Test Accuracy')

# Optimizer
opt_data = summary_df[summary_df['Kategori'] == 'Optimizer'].sort_values('Test Accuracy')
axes[2].barh(opt_data['Parametre'], opt_data['Test Accuracy'], color='mediumseagreen')
axes[2].set_title('Optimizerlar', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Test Accuracy')

plt.suptitle('Kategoriye Göre En İyi Sonuçlar', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Sonuç ve Öneriler

### Aktivasyon Fonksiyonları:
- **ReLU**: Genellikle hızlı öğrenme ve iyi performans
- **Sigmoid**: Küçük veri setlerinde iş görebilir ancak vanishing gradient sorunu
- **Tanh**: Sigmoid'den daha iyi ama aynı sorunlar

### Epoch Sayısı:
- Çok düşük epoch (20) yetersiz öğrenme
- Çok yüksek epoch (200+) overfitting riski
- Early stopping kullanımı önerilir

### Optimizerlar:
- **Adam**: Genel olarak en iyi performans
- **Nadam**: Adam + Nesterov momentum
- **RMSprop**: RNN için iyi ancak MLP'de de etkili
- **SGD**: Yavaş ama daha kararlı

### Önerilen Yapılandırma:
- Aktivasyon: ReLU
- Optimizer: Adam
- Epoch: 50-100 (Early stopping ile)
- Batch Size: 16-32
- Learning Rate: 0.001
- Dropout: 0.2-0.3
- BatchNormalization: Kullan